# RBI Mule Account Detection — EDA & Data Quality

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from src.data.loader import load_all
from src.data.validator import DataValidator

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

In [ ]:
txn, static = load_all()
print(f'Transactions: {txn.shape}')
for k, v in static.items():
    print(f'  {k}: {v.shape if v is not None else "MISSING"}')

## Data Quality Report

In [ ]:
validator = DataValidator()
report = validator.run_full_validation(txn, static)

# Null percentages bar chart
if 'transactions' in report and 'null_pct' in report['transactions']:
    null_pct = pd.Series(report['transactions']['null_pct'])
    null_pct = null_pct[null_pct > 0].sort_values()
    if len(null_pct) > 0:
        null_pct.plot.barh(figsize=(10, 5), title='Null Percentages by Column (%)')
        plt.xlabel('Null %')
        plt.tight_layout()
        plt.show()
    else:
        print('No null values found!')

## Label Distribution

In [ ]:
labels = static.get('labels')
if labels is not None and 'is_mule' in labels.columns:
    counts = labels['is_mule'].value_counts().rename({0: 'Legitimate', 1: 'Mule'})
    print(f'Mule: {counts.get("Mule", 0)} ({counts.get("Mule", 0)/len(labels)*100:.2f}%)')
    print(f'Legitimate: {counts.get("Legitimate", 0)} ({counts.get("Legitimate", 0)/len(labels)*100:.2f}%)')
    counts.plot.bar(color=['steelblue', 'crimson'], title='Label Distribution')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## Transaction Volume Over Time

In [ ]:
if not txn.empty and 'transaction_date' in txn.columns:
    txn['month'] = txn['transaction_date'].dt.to_period('M')
    monthly = txn.groupby('month').size()
    monthly.index = monthly.index.astype(str)
    monthly.plot(figsize=(12, 4), title='Monthly Transaction Count', marker='o')
    plt.ylabel('Transaction Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Amount Distribution

In [ ]:
if not txn.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    txn['transaction_amount'].clip(upper=txn['transaction_amount'].quantile(0.99)).hist(
        bins=100, ax=ax, alpha=0.7, log=True)
    ax.set_xlabel('Transaction Amount')
    ax.set_ylabel('Count (log scale)')
    ax.set_title('Transaction Amount Distribution')
    plt.tight_layout()
    plt.show()

## Temporal Patterns

In [ ]:
if not txn.empty and 'transaction_date' in txn.columns:
    txn_temp = txn.copy()
    txn_temp['hour'] = txn_temp['transaction_date'].dt.hour
    txn_temp['dow'] = txn_temp['transaction_date'].dt.day_name()
    heatmap_data = txn_temp.groupby(['dow', 'hour']).size().unstack(fill_value=0)
    dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    heatmap_data = heatmap_data.reindex([d for d in dow_order if d in heatmap_data.index])
    plt.figure(figsize=(14, 5))
    sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False)
    plt.title('Transaction Count by Hour of Day and Day of Week')
    plt.xlabel('Hour of Day')
    plt.ylabel('Day of Week')
    plt.tight_layout()
    plt.show()

## Key Findings

- **Data quality**: [Fill after running cells]
- **Label imbalance**: [Fill after running cells]
- **Temporal patterns**: [Fill after running cells]
- **Amount distribution**: [Fill after running cells]
- **Missing data**: [Fill after running cells]